In [169]:
!pip install colorize_pinyin
!pip install pinyin-tone-converter
!pip install dragonmapper
!pip install edge-tts

In [1]:
import json
import re
import os
import colorize_pinyin
from pinyin_tone_converter.pinyin_tone_converter import PinyinToneConverter
from dragonmapper import transcriptions, hanzi

In [2]:
def load_frequency_data(freq_file_path):
    """Load frequency data from the frequency file"""
    freq_dict = {}
    
    try:
        with open(freq_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if '\t' in line:
                    parts = line.split('\t')
                    if len(parts) >= 2:
                        char = parts[0]
                        try:
                            frequency = int(parts[1])
                            freq_dict[char] = frequency
                        except ValueError:
                            # Skip lines where frequency is not a number
                            continue
    except FileNotFoundError:
        print(f"Error: Frequency file not found at {freq_file_path}")
        return None
    except Exception as e:
        print(f"Error reading frequency file: {e}")
        return None
    
    return freq_dict

freq_dict = load_frequency_data("HSK-3.0-words-list/Scripts and data/blog_lit_news_tech_weibo_freq.release_sorted.txt")

In [3]:
with open("HSK-3.0-words-list/Scripts and data/all_cedict.json", "r", encoding="utf-8") as f:
    dictionary_data = json.load(f)

In [4]:
w = "不予" #"新能源" #"新媒体" #"偶然" #"电动车" # "大赛" # "检票"
d = dictionary_data.get(w, [])
d

{'simplified': '不予',
 'traditional': '不予',
 'pinyin': ['bu4 yu3'],
 'definitions': {'bu4 yu3': 'to not give; to refuse; to withhold; '}}

In [5]:
pinyin = "bia1" #"ce4 lu:e4"
if "u:" in pinyin:
    pinyin = pinyin.replace("u:", "v")
PinyinToneConverter().convert_text(pinyin)

'biā'

In [6]:
TONES_CLASSES = (u"text-color5", u"text-color1", u"text-color2", u"text-color3", u"text-color4")

def get_pinyin_html(pinyin):
    if "u:" in pinyin:
        pinyin = pinyin.replace("u:", "v")
        
    converted_pinyin = PinyinToneConverter().convert_text(pinyin)
    # print("Converted pinyin:", converted_pinyin)
    pinyin_html = colorize_pinyin.colorized_HTML_string_from_string(converted_pinyin, pinyin_wrapper_class="pinYinWrapper", tones_classes=TONES_CLASSES)

    if not pinyin_html:
        pinyin_html = f"<span class=\"pinYinWrapper\"><span class=\"text-color5\">{converted_pinyin}</span></span>"
    return pinyin_html

print(get_pinyin_html("nǐ hǎo"))
print(get_pinyin_html("de5"))
print(get_pinyin_html("ce4 lu:e4"))
print(get_pinyin_html("bia1"))

<span class="pinYinWrapper"><span class="text-color3">nǐ</span> <span class="text-color3">hǎo</span></span>
<span class="pinYinWrapper"><span class="text-color5">de</span></span>
<span class="pinYinWrapper"><span class="text-color4">cè</span> <span class="text-color4">lüè</span></span>
<span class="pinYinWrapper"><span class="text-color5">bi</span><span class="text-color1">ā</span></span>


In [7]:
definitions = dictionary_data.get("有点儿", [])

for pinyin in definitions['definitions']:
    print("===================================")
    print(f"Pinyin: {pinyin}")
    color_pinyin = get_pinyin_html(pinyin)
    print(color_pinyin)

    definition = definitions['definitions'][pinyin]
    definition = definition.split("; ")

    defn_html = "<ul>\n"

    for defn in definition:
        defn = defn.strip()
        if defn:
            defn_html += f"\t<li>{defn}</li>\n"
    defn_html += "</ul>"
    print(defn_html)

    # print(f"Pinyin: {pinyin}, Definition: {definition}")


Pinyin: you3 dian3 r5
<span class="pinYinWrapper"><span class="text-color3">yǒu</span> <span class="text-color3">diǎn</span> <span class="text-color5">r</span>5</span>
<ul>
	<li>slightly</li>
	<li>a little</li>
	<li>somewhat</li>
</ul>


# Get Colorized Hanzi Pinyin

In [8]:
from posixpath import sep


def get_colorized_hanzi_pinyin(word):
    definitions = dictionary_data.get(word, [])

    charsim = definitions.get('simplified', word)
    chartrad = definitions.get('traditional', word)

    html = ""
    variant_html = ""

    for pinyin in definitions['definitions']:
        # print("===================================")
        # print(f"Pinyin: {pinyin}")
        color_pinyin = get_pinyin_html(pinyin)
        # print(color_pinyin)

        color_list = re.findall(r'<span class="text-color\d">.*?<\/span>', color_pinyin)

        # print("Color List:", color_list)
        
        # Simplified
        # replace pinyin with hanzi in word
        for i, pinyin_part in enumerate(color_list):
            hanzi_char = word[i]
            color_list[i] = pinyin_part.replace(re.search(r'>(.*?)<', pinyin_part).group(1), hanzi_char)

        # print("Color List:", "".join(color_list))

        html += '<div class="char">\n\t<span id="char-sim-id">' + "".join(color_list) + "\n</span>\n"
        
        # Traditional
        if charsim != chartrad:
            for i, pinyin_part in enumerate(color_list):
                hanzi_char = definitions['traditional'][i]
                color_list[i] = pinyin_part.replace(re.search(r'>(.*?)<', pinyin_part).group(1), hanzi_char)

            html += '<span class="sep">〔</span><span id="char-trad-id">\n\t' + "".join(color_list) + '\n</span><span class="sep">〕</span>'
        
        html += "</div>\n"

        definition = definitions['definitions'][pinyin]
        definition = definition.split("; ")

        html += color_pinyin + "\n"

        defn_html = "<ul>\n"
        for defn in definition:
            defn = defn.strip()
            if defn:
                # print("Definition:", defn)

                # if "variant of" in defn:
                #     defn = defn.strip()
                #     defn_html += f"\t<li>{defn}</li>\n"

                #     w = defn.split("variant of")[-1].split("[")[0].strip()
                #     # print("Original variant word:", w)

                #     if "|" in w:
                #         w = w.split("|")

                #         if len(w) > 1:
                #             w = w[1].strip()
                    
                    
                #     # print("Variant word:", w)

                #     definition1 = dictionary_data.get(w, [])
                #     # print("Definition1:", definition1)

                #     for piny1 in definition1['definitions']:
                #         defn1 = definition1['definitions'][piny1]
                #         defn1 = defn1.split("; ")
                #         color_pinyin1 = get_pinyin_html(piny1)

                #         # print("Color Pinyin1:", color_pinyin1)

                #         color_list1 = re.findall(r'<span class="t\d">.*?<\/span>', color_pinyin1)

                #         # Simplified
                #         # replace pinyin with hanzi in word
                #         for i, pinyin_part in enumerate(color_list1):
                #             hanzi_char = word[i]
                #             color_list1[i] = pinyin_part.replace(re.search(r'>(.*?)<', pinyin_part).group(1), hanzi_char)
                        
                #         variant_html += "<div class='simplified'>\n\t" + "".join(color_list1) + "\n</div>\n"
                        

                #         # Traditional
                #         # replace pinyin with hanzi in word
                #         for i, pinyin_part in enumerate(color_list1):
                #             # hanzi_char = word[i]
                #             hanzi_char = definition1['traditional'][i]
                #             color_list1[i] = pinyin_part.replace(re.search(r'>(.*?)<', pinyin_part).group(1), hanzi_char)
                        
                #         variant_html += "<div class='traditional'>\n\t" + "".join(color_list1) + "\n</div>\n"

                #         variant_html += color_pinyin1 + "\n"

                #         def_variant_html = "<ul>\n"
                #         for defn1 in defn1:
                #             defn1 = defn1.strip()
                #             if defn1:
                #                 def_variant_html += f"\t<li>{defn1}</li>\n"
                #     def_variant_html += "</ul>\n"
                #     variant_html += def_variant_html + "\n"
                #     # print("Variant HTML:", variant_html)

                # else:
                    # defn_html += f"\t<li>{defn}</li>\n"
                defn_html += f"\t<li>{defn}</li>\n"
        defn_html += "</ul>\n"

        html += defn_html + "\n"
        # html += variant_html + "\n"

        # print(defn_html)
        # print(f"Pinyin: {pinyin}, Definition: {definition}")
    
    return charsim, chartrad, html

# defn_html = get_colorized_hanzi_pinyin("有点儿")
# print(defn_html)

# defn_html = get_colorized_hanzi_pinyin("的")
# print(defn_html)

# Level: 1, Word: 一点儿, Pinyin: yìdiǎnr, POS: 副, Frequency: 129371, Dict Data: <span class="t1">一</span><span class="t3">点</span><span class="t0">儿</span>
# <span class="pinYinWrapper"><span class="t1">yī</span> <span class="t3">diǎn</span> <span class="t0">r</span>5</span>
# <ul>
# 	<li>erhua variant of 一點|一点[yi1 dian3]</li>
# </ul>

# defn_html = get_colorized_hanzi_pinyin("坐")
# print(defn_html)

# defn_html = get_colorized_hanzi_pinyin("一点儿")
# print(defn_html)

# defn_html.replace("\n", " ").replace("\t", " ")

# defn_html = get_colorized_hanzi_pinyin("男孩儿")
# print(defn_html)

# defn_html = get_colorized_hanzi_pinyin("检票")
# print(defn_html)

# defn_html = get_colorized_hanzi_pinyin("大赛")
# print(defn_html)

# simplified, traditional, defn_html = get_colorized_hanzi_pinyin("电动车")

# simplified, traditional, defn_html = get_colorized_hanzi_pinyin("策略")
# print("Simplified:", simplified)
# print("Traditional:", traditional)
# print("Definition HTML:", defn_html)

simplified, traditional, defn_html = get_colorized_hanzi_pinyin("吧")
print("Simplified:", simplified)
print("Traditional:", traditional)
print("Definition HTML:", defn_html)

Simplified: 吧
Traditional: 吧
Definition HTML: <div class="char">
	<span id="char-sim-id"><span class="text-color1">吧</span>
</span>
</div>
<span class="pinYinWrapper"><span class="text-color1">bā</span></span>
<ul>
	<li>bar (loanword) (serving drinks, or providing Internet access etc)</li>
	<li>to puff (on a pipe etc)</li>
	<li>(onom.) bang</li>
	<li>abbr. for 貼吧|贴吧[tie1 ba1]</li>
</ul>

<div class="char">
	<span id="char-sim-id"><span class="text-color5">吧</span>
</span>
</div>
<span class="pinYinWrapper"><span class="text-color5">ba</span></span>
<ul>
	<li>(modal particle indicating suggestion or surmise)</li>
	<li>...right?</li>
	<li>...OK?</li>
	<li>...I presume.</li>
</ul>




In [9]:
import re

def split_variants(word, pinyin):
    """Return two (word, pinyin) variants for a string with parentheses."""
    m_word = re.search(r'[(（]([^）\)]*)[)）]', word)
    m_pin = re.search(r'[(（]([^）\)]*)[)）]', pinyin)

    if not m_word:
        return [(word, pinyin)]

    parent = m_word.group(1)
    w_before = word[:m_word.start()]
    w_after = word[m_word.end():]
    p_parent = m_pin.group(1) if m_pin else ''
    p_before = pinyin[:m_pin.start()] if m_pin else pinyin
    p_after = pinyin[m_pin.end():] if m_pin else ''

    # If there's text after the parentheses, the parent typically replaces the text before the parentheses
    if w_after:
        variant1 = (w_before + w_after, (p_before + p_after).strip())
        variant2 = (parent + w_after, (p_parent + p_after).strip())
    else:
        # If nothing follows the parentheses, the parent is usually appended to the prefix
        variant1 = (w_before, p_before.strip())
        variant2 = (w_before + parent, (p_before + p_parent).strip())

    return [variant1, variant2]

# Examples
examples = [
    ("有（一）点儿", "yǒu(yì)diǎnr"),
    ("没（有）", "méi(yǒu)")
]

for word, pinyin in examples:
    for w, p in split_variants(word, pinyin):
        print(w, "-", p)


有点儿 - yǒudiǎnr
一点儿 - yìdiǎnr
没 - méi
没有 - méiyǒu


# Create Anki HTML

In [10]:
level = "1"

fname = f"HSK-3.0-words-list/New HSK (2025)/HSK Words/temp/HSK_Level_{level}_words.txt"

with open(fname, "r", encoding="utf-8") as f:
    lines = f.readlines()

words = ["有（一）点儿", "没（有）", "有时（候）", "差（一）点儿", "要不（然）", "凡（是）"]
pinyins = ["yǒu(yì)diǎnr", "méi(yǒu)", "yǒushí(hou)", "chà(yì)diǎnr", "yàobù(rán)", "fán(shì)"]

for word, pinyin in zip(words, pinyins):
    for line in lines:
        if word in line:
            print(line)
            lines.remove(line)

            for w, p in split_variants(word, pinyin):
                new_line = line.replace(word, w).replace(pinyin, p)
                print("Adding:", new_line.strip())
                lines.append(new_line)

anki_data = {}

for line in lines:
    l = line.strip().split("\t")
    # print(l)

    level = l[1]
    word = l[2]

    # if word contains number then remove it
    word = ''.join([i for i in word if not i.isdigit()])

    pinyin = l[3]
    # print("Original Pinyin:", pinyin)
    zhuyin = hanzi.to_zhuyin(word)
    
    if len(l) > 4:
        pos = l[4]
    else:
        pos = ""
    
    freq = freq_dict.get(word, 0) if freq_dict else 0

    print(f"Processing word: {word}, Pinyin: {pinyin}, Level: {level}, POS: {pos}, Frequency: {freq}")

    # defn_html = get_colorized_hanzi_pinyin(word)
    simplified, traditional, defn_html = get_colorized_hanzi_pinyin(word)

    # if definitions == []:
    #     dict_data = "N/A"
    #     print("No dictionary data found for word:", word)

    # remove new line and tab from defn_html
    defn_html = defn_html.replace("\n", " ").replace("\t", " ")

    # print(f"Level: {level}, Word: {word}, Pinyin: {pinyin}, POS: {pos}, Frequency: {freq}, Dict Data: {defn_html}")

    anki_data[word] = {
        "simplified": simplified,
        "traditional": traditional,
        "level": level,
        "pinyin": pinyin,
        "zhuyin": zhuyin,
        "pos": pos,
        "frequency": freq,
        "definition_html": defn_html
    }

# sort anki_data by frequency descending
anki_data_sorted = dict(sorted(anki_data.items(), key=lambda item: item[1]['frequency'], reverse=True))

# save as tab separated file
with open(f"HSK-3.0-words-list/New HSK (2025)/Anki xiehanzi/HSK_Level_{level}.txt", "w", encoding="utf-8") as f:
    for word, data in anki_data_sorted.items():
        f.write(f"{word}\t{data['traditional']}\t{data['pinyin']}\t{data['zhuyin']}\t{data['level']}\t{data['pos']}\t{data['frequency']}\t{data['definition_html']}\n")

266	1	有（一）点儿	yǒu(yì)diǎnr	副

Adding: 266	1	有点儿	yǒudiǎnr	副
Adding: 266	1	一点儿	yìdiǎnr	副
123	1	没（有）	méi(yǒu)	动、副

Adding: 123	1	没	méi	动、副
Adding: 123	1	没有	méiyǒu	动、副
Processing word: 爱, Pinyin: ài, Level: 1, POS: 动, Frequency: 13772657
Processing word: 吧, Pinyin: ba, Level: 1, POS: 助, Frequency: 17812571
Processing word: 八, Pinyin: bā, Level: 1, POS: 数, Frequency: 0
Processing word: 爸爸, Pinyin: bàba, Level: 1, POS: 名, Frequency: 1539349
Processing word: 百, Pinyin: bǎi, Level: 1, POS: 数, Frequency: 0
Processing word: 白天, Pinyin: báitiān, Level: 1, POS: 名, Frequency: 606766
Processing word: 半, Pinyin: bàn, Level: 1（4）, POS: 数、（副）, Frequency: 2854116
Processing word: 包子, Pinyin: bāozi, Level: 1, POS: 名, Frequency: 180316
Processing word: 杯子, Pinyin: bēizi, Level: 1, POS: 名, Frequency: 195418
Processing word: 本, Pinyin: běn, Level: 1, POS: 量, Frequency: 15546838
Processing word: 边, Pinyin: biān, Level: 1, POS: 名、后缀, Frequency: 3385425
Processing word: 病, Pinyin: bìng, Level: 1, POS: 名、动, Freq

In [11]:
import edge_tts

VOICE = "zh-CN-XiaoxiaoNeural"

def save_audio(word) -> None:
    text = word.strip()
    output_file = f"HSK-3.0-words-list/New HSK (2025)/Audio/cmn-{text}.mp3"
    # output_file = f"cmn-{word}.mp3"
    communicate = edge_tts.Communicate(text, VOICE)
    communicate.save_sync(output_file)

save_audio("你好")

In [12]:
os.path.exists("HSK-3.0-words-list/New HSK (2025)/Audio/cmn-证券.mp3")

True

In [13]:
# get zero length files in the audio folder
audio_folder = "HSK-3.0-words-list/New HSK (2025)/Audio/"
zero_length_files = [f for f in os.listdir(audio_folder) if os.path.getsize(os.path.join(audio_folder, f)) == 0]

# remove zero length files
for f in zero_length_files:
    os.remove(os.path.join(audio_folder, f))
    print(f"Removed zero length file: {f}")

In [14]:
level = "7-9"
fname = f"HSK-3.0-words-list/New HSK (2025)/Anki xiehanzi/HSK_Level_{level}.txt"

with open(fname, "r", encoding="utf-8") as f:
    data = f.read().strip().split("\n")
    for line in data:
        # /Users/mani/Library/Application Support/Anki2/User 2/collection.media/
        audio_path = "/Users/mani/Library/Application Support/Anki2/User 2/collection.media/"
        audio = "cmn-{}.mp3".format(line.split("\t")[0])
        # print(audio)

        if os.path.exists("HSK-3.0-words-list/New HSK (2025)/Audio/" + audio):
            continue

        if not os.path.exists(audio_path + audio):
            print(f"Audio not found for {line}")
            # use edge-tts to generate audio

            save_audio(line.split("\t")[0])

        else:
            # copy to "HSK-3.0-words-list/New HSK (2025)/Audio"
            os.system(f'cp "{audio_path + audio}" "HSK-3.0-words-list/New HSK (2025)/Audio/"')

# Anki Deck

# Audio Template - Card 4

In [15]:
import genanki
import random

In [16]:
def create_audio_deck():
  with open("card templates/Card 4/front.html", "r") as f:
      card_template_front_html = f.read()

  with open("card templates/Card 4/back.html", "r") as f:
      card_template_back_html = f.read()

  with open("card templates/styling-xiehanzi-3.0.css", "r") as f:
      card_styles = f.read()

  anki_model = genanki.Model(
    model_id=random.randrange(1 << 30, 1 << 31),
    name='Basic - New HSK (2025)',
    # ["Simplified", "Traditional", "Pinyin", "Zhuyin", "PoS", "Meaning", "Audio"],
    fields=[
      {'name': 'Simplified'},
      {'name': 'Traditional'},
      {'name': 'Pinyin'},
      {'name': 'Zhuyin'},
      {'name': 'PoS'},
      {'name': 'Meaning'},
      {'name': 'Audio'},
    ],
    templates=[
      {
        'name': 'Card 1 - Audio',
        'qfmt': card_template_front_html,
        'afmt': card_template_back_html,
      },
    ],
    css=card_styles
    )

  sub_deck = genanki.Deck(random.randrange(1 << 30, 1 << 31), f'Anki Xiehanzi - New HSK (2025)::HSK {level}::Audio')

  with open(fname, "r", encoding="utf-8") as f:
      lines = f.readlines()

      for line in lines:
          l = line.strip().split("\t")

          simplified = l[0]
          traditional = l[1]
          pinyin = l[2]
          zhuyin = l[3]
          pos = l[5]
          definition_html = l[7]

          fields_list = [
              simplified,  # Simplified
              traditional,  # Traditional
              pinyin,  # Pinyin
              zhuyin,  # Zhuyin
              pos,  # PoS
              definition_html,  # Definition
              f"[sound:cmn-{simplified}.mp3]"  # Audio
          ]

          note = genanki.Note(model=anki_model, fields=fields_list)
          sub_deck.add_note(note)
  return sub_deck

# Meaning Template -  Card 1

In [17]:
def create_meaning_deck():
  with open("card templates/Card 1/front.html", "r") as f:
      card_template_front_html = f.read()

  with open("card templates/Card 1/back.html", "r") as f:
      card_template_back_html = f.read()

  with open("card templates/styling-xiehanzi-3.0.css", "r") as f:
      card_styles = f.read()

  anki_model = genanki.Model(
    model_id=random.randrange(1 << 30, 1 << 31),
    name='Basic - New HSK (2025)',
    # ["Simplified", "Traditional", "Pinyin", "Zhuyin", "PoS", "Meaning", "Audio"],
    fields=[
      {'name': 'Simplified'},
      {'name': 'Traditional'},
      {'name': 'Pinyin'},
      {'name': 'Zhuyin'},
      {'name': 'PoS'},
      {'name': 'Meaning'},
      {'name': 'Audio'},
    ],
    templates=[
      {
        'name': 'Card 1 - Meaning',
        'qfmt': card_template_front_html,
        'afmt': card_template_back_html,
      },
    ],
    css=card_styles
    )

  sub_deck = genanki.Deck(random.randrange(1 << 30, 1 << 31), f'Anki Xiehanzi - New HSK (2025)::HSK {level}::Meaning')

  with open(fname, "r", encoding="utf-8") as f:
      lines = f.readlines()

      for line in lines:
          l = line.strip().split("\t")

          simplified = l[0]
          traditional = l[1]
          pinyin = l[2]
          zhuyin = l[3]
          pos = l[5]
          definition_html = l[7]

          fields_list = [
              simplified,  # Simplified
              traditional,  # Traditional
              pinyin,  # Pinyin
              zhuyin,  # Zhuyin
              pos,  # PoS
              definition_html,  # Definition
              f"[sound:cmn-{simplified}.mp3]"  # Audio
          ]

          note = genanki.Note(model=anki_model, fields=fields_list)
          sub_deck.add_note(note)

  return sub_deck

# Pinyin Template - Card 2

In [18]:
def create_pinyin_deck():
  with open("card templates/Card 2/front.html", "r") as f:
      card_template_front_html = f.read()

  with open("card templates/Card 2/back.html", "r") as f:
      card_template_back_html = f.read()

  with open("card templates/styling-xiehanzi-3.0.css", "r") as f:
      card_styles = f.read()

  anki_model = genanki.Model(
    model_id=random.randrange(1 << 30, 1 << 31),
    name='Basic - New HSK (2025)',
    # ["Simplified", "Traditional", "Pinyin", "Zhuyin", "PoS", "Meaning", "Audio"],
    fields=[
      {'name': 'Simplified'},
      {'name': 'Traditional'},
      {'name': 'Pinyin'},
      {'name': 'Zhuyin'},
      {'name': 'PoS'},
      {'name': 'Meaning'},
      {'name': 'Audio'},
    ],
    templates=[
      {
        'name': 'Card 1 - Pinyin',
        'qfmt': card_template_front_html,
        'afmt': card_template_back_html,
      },
    ],
    css=card_styles
    )

  sub_deck = genanki.Deck(random.randrange(1 << 30, 1 << 31), f'Anki Xiehanzi - New HSK (2025)::HSK {level}::Pinyin')

  with open(fname, "r", encoding="utf-8") as f:
      lines = f.readlines()

      for line in lines:
          l = line.strip().split("\t")

          simplified = l[0]
          traditional = l[1]
          pinyin = l[2]
          zhuyin = l[3]
          pos = l[5]
          definition_html = l[7]

          fields_list = [
              simplified,  # Simplified
              traditional,  # Traditional
              pinyin,  # Pinyin
              zhuyin,  # Zhuyin
              pos,  # PoS
              definition_html,  # Definition
              f"[sound:cmn-{simplified}.mp3]"  # Audio
          ]

          note = genanki.Note(model=anki_model, fields=fields_list)
          sub_deck.add_note(note)

  return sub_deck

# Write Template - Card 5

In [19]:
def create_write_deck():
  with open("card templates/Card 5/front-xiehanzi-3.0.html", "r") as f:
      card_template_front_html = f.read()

  with open("card templates/Card 5/back.html", "r") as f:
      card_template_back_html = f.read()

  with open("card templates/styling-xiehanzi-3.0.css", "r") as f:
      card_styles = f.read()

  anki_model = genanki.Model(
    model_id=random.randrange(1 << 30, 1 << 31),
    name='Basic - New HSK (2025)',
    # ["Simplified", "Traditional", "Pinyin", "Zhuyin", "PoS", "Meaning", "Audio"],
    fields=[
      {'name': 'Simplified'},
      {'name': 'Traditional'},
      {'name': 'Pinyin'},
      {'name': 'Zhuyin'},
      {'name': 'PoS'},
      {'name': 'Meaning'},
      {'name': 'Audio'},
    ],
    templates=[
      {
        'name': 'Card 1 - Write',
        'qfmt': card_template_front_html,
        'afmt': card_template_back_html,
      },
    ],
    css=card_styles
    )

  sub_deck = genanki.Deck(random.randrange(1 << 30, 1 << 31), f'Anki Xiehanzi - New HSK (2025)::HSK {level}::Write')

  with open(fname, "r", encoding="utf-8") as f:
      lines = f.readlines()

      for line in lines:
          l = line.strip().split("\t")

          simplified = l[0]
          traditional = l[1]
          pinyin = l[2]
          zhuyin = l[3]
          pos = l[5]
          definition_html = l[7]

          fields_list = [
              simplified,  # Simplified
              traditional,  # Traditional
              pinyin,  # Pinyin
              zhuyin,  # Zhuyin
              pos,  # PoS
              definition_html,  # Definition
              f"[sound:cmn-{simplified}.mp3]"  # Audio
          ]

          note = genanki.Note(model=anki_model, fields=fields_list)
          sub_deck.add_note(note)

  return sub_deck

# Media

In [20]:
media_files = [
    # fonts
    "card templates/fonts/_MaterialIcons-Regular.woff",
    "card templates/fonts/_MaterialIcons-Regular.woff2",
    
    # files
    "card templates/files/_pleco.png",
    "card templates/files/_youdao.png",
    "card templates/files/_rtega.png",
    "card templates/files/_tatoeba.png",
    "card templates/files/_hanzicraft.png",
    "card templates/files/_characterpop.svg"
]

# Generate

In [21]:
decks = []

level = "1"

for i in range(1, 8):
    if i == 7:
        level = "7-9"
    else:
        level = str(i)

    fname = f"HSK-3.0-words-list/New HSK (2025)/Anki xiehanzi/HSK_Level_{level}.txt"

    audio_deck = create_audio_deck()
    meaning_deck = create_meaning_deck()
    pinyin_deck = create_pinyin_deck()
    write_deck = create_write_deck()

    decks.append(audio_deck)
    decks.append(meaning_deck)
    decks.append(pinyin_deck)
    decks.append(write_deck)

    with open(fname, "r", encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines:
            l = line.strip().split("\t")
            simplified = l[0]
            audio_file = f"HSK-3.0-words-list/New HSK (2025)/Audio/cmn-{simplified}.mp3"
            media_files.append(audio_file)

genanki.Package(decks, media_files=media_files).write_to_file(f'Anki-xiehanzi - New HSK (2025).apkg')

/Users/mani/miniconda3/lib/python3.13/site-packages/genanki/note.py:148: UserWarning: Field contained the following invalid HTML tags. Make sure you are calling html.escape() if your field data isn't already HTML-encoded: <sp啊n cl啊ss="text-color5"> </sp啊n>
  warnings.warn("Field contained the following invalid HTML tags. Make sure you are calling html.escape() if"
/Users/mani/miniconda3/lib/python3.13/site-packages/genanki/note.py:148: UserWarning: Field contained the following invalid HTML tags. Make sure you are calling html.escape() if your field data isn't already HTML-encoded: <</li>
  warnings.warn("Field contained the following invalid HTML tags. Make sure you are calling html.escape() if"
/Users/mani/miniconda3/lib/python3.13/site-packages/genanki/note.py:148: UserWarning: Field contained the following invalid HTML tags. Make sure you are calling html.escape() if your field data isn't already HTML-encoded: <spa虐 class="text-color5"> </spa虐>
  warnings.warn("Field contained the 

# Anki Deck with Example Sentences

In [20]:
import genanki
import random

In [21]:
def create_deck(card_type, card_number, level, fname):
    card_paths = {
        1: "card templates/Card 1",
        2: "card templates/Card 2", 
        4: "card templates/Card 4",
        5: "card templates/Card 5"
    }
    
    card_path = card_paths.get(card_number, "card templates/Card 1")
    
    # new front for write, else use old type for other
    front_file = "card templates/New/front-xiehanzi-3.0.html" if card_type.lower() == "write" else f"{card_path}/front.html"
    
    # new backside template
    back_file = f"{card_path}/back.html" if card_type.lower() == "write" else "card templates/New/back.html"
    
    with open(front_file, "r") as f:
        card_template_front_html = f.read()

    with open(back_file, "r") as f:
        card_template_back_html = f.read()

    with open("card templates/New/styling.css", "r") as f:
        card_styles = f.read()

    anki_model = genanki.Model(
        model_id=random.randrange(1 << 30, 1 << 31),
        name=f'Basic - New HSK (2025) with sentence - {card_type}',
        fields=[
            {'name': 'ID'},
            {'name': 'Simplified'},
            {'name': 'Traditional'},
            {'name': 'Pinyin'},
            {'name': 'Zhuyin'},
            {'name': 'PoS'},
            {'name': 'Meaning'},
            {'name': 'Audio'},
        ],
        templates=[
            {
                'name': f'Card {card_number} - {card_type}',
                'qfmt': card_template_front_html,
                'afmt': card_template_back_html,
            },
        ],
        css=card_styles
    )

    sub_deck = genanki.Deck(
        random.randrange(1 << 30, 1 << 31), 
        f'Anki-xiehanzi - New HSK (2025) with sentences::HSK {level}::{card_type}'
    )

    with open(fname, "r", encoding="utf-8") as f:
        lines = f.readlines()

        for line in lines:
            l = line.strip().split("\t")

            simplified = l[0]
            traditional = l[1]
            pinyin = l[2]
            zhuyin = l[3]
            pos = l[5]
            definition_html = l[7]

            fields_list = [
                str(simplified+pinyin),
                simplified,
                traditional,
                pinyin,
                zhuyin,
                pos,
                definition_html,
                f"[sound:cmn-{simplified}.mp3]"
            ]

            note = genanki.Note(model=anki_model, fields=fields_list)
            sub_deck.add_note(note)

    return sub_deck

In [22]:
media_files = [
    # fonts
    "card templates/fonts/_MaterialIcons-Regular.woff",
    "card templates/fonts/_MaterialIcons-Regular.woff2",
    
    # files
    "card templates/files/_pleco.png",
    "card templates/files/_youdao.png",
    "card templates/files/_rtega.png",
    "card templates/files/_tatoeba.png",
    "card templates/files/_hanzicraft.png",
    "card templates/files/_characterpop.svg",

    # sql db
    "card templates/files/_chinese_sentences.db"
]

In [23]:
decks = []

card_types = [
    ("Audio", 4),
    ("Meaning", 1),
    ("Pinyin", 2),
    ("Write", 5)
]

for i in range(1, 8):
    level = "7-9" if i == 7 else str(i)
    fname = f"HSK-3.0-words-list/New HSK (2025)/Anki xiehanzi/HSK_Level_{level}.txt"

    for card_type, card_number in card_types:
        deck = create_deck(card_type, card_number, level, fname)
        decks.append(deck)

    with open(fname, "r", encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines:
            l = line.strip().split("\t")
            simplified = l[0]
            audio_file = f"HSK-3.0-words-list/New HSK (2025)/Audio/cmn-{simplified}.mp3"
            media_files.append(audio_file)

genanki.Package(decks, media_files=media_files).write_to_file(f'Anki-xiehanzi - New HSK (2025) with sentences.apkg')